#### Incremental ingest (new batch)

Generates a NEW batch of construction data (same distribution, different
records), lands it in its own dated bronze folder, cleans it with the same
silver logic, and APPENDS to the silver Delta tables with a dedup safety net.

Purpose: produce unseen data to test the ML models on records they weren't
trained on; the honest way to check a model generalizes.

#### Cell 1

In [7]:
# Install + seed Faker
%pip install faker

StatementMeta(, 6118298a-33bb-43a3-9cda-5976b95a5e7a, 26, Finished, Available, Finished, False)


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



#### Cell 2

In [8]:
# Batch config (auto-detects next batch from the Lakehouse)
#
# Leave BATCH_ID_OVERRIDE = None to auto-detect the next batch number from
# what's already in silver (B1 = original load, so first batch here is B2).
# Set it to a specific value (e.g. "B2") to deliberately regenerate/replace
# that batch instead -- the dedup safety net makes that idempotent.
BATCH_ID_OVERRIDE = None      # e.g. "B2" to regenerate a specific batch
N_PROJECTS        = 40        # size of the new batch

import os, random
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
from faker import Faker

# calendar-safe parse + write (same settings silver needs)
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")

def next_batch_id():
    # silver not built yet -> first incremental batch is B2 (B1 = original load)
    if not spark.catalog.tableExists("silver_projects"):
        return "B2"
    existing = spark.table("silver_projects").select("batch_id").distinct().collect()
    nums = []
    for row in existing:
        bid = row["batch_id"]
        if bid and bid.startswith("B") and bid[1:].isdigit():
            nums.append(int(bid[1:]))
    return f"B{(max(nums) + 1) if nums else 2}"

BATCH_ID = BATCH_ID_OVERRIDE if BATCH_ID_OVERRIDE else next_batch_id()
SEED     = 200 + int(BATCH_ID[1:])      # deterministic seed per batch -> reproducible

random.seed(SEED); np.random.seed(SEED)

BRONZE_BATCH       = f"Files/bronze/construction_raw/batch_{BATCH_ID}"
BRONZE_BATCH_LOCAL = f"/lakehouse/default/{BRONZE_BATCH}"
os.makedirs(BRONZE_BATCH_LOCAL, exist_ok=True)

print(f"Batch {BATCH_ID} | seed {SEED} | {N_PROJECTS} projects -> {BRONZE_BATCH}")

StatementMeta(, 6118298a-33bb-43a3-9cda-5976b95a5e7a, 28, Finished, Available, Finished, False)

Batch B2 | seed 202 | 40 projects -> Files/bronze/construction_raw/batch_B2


#### Cell 3

In [9]:
# Generator (same logic as bronze, with BATCH_ID-prefixed IDs)
Faker.seed(SEED)
fake = Faker()

PROJECT_TYPES = ["Healthcare","Commercial","Higher Education","K-12 Education",
    "Government/Civic","Aviation","Data Center","Mission Critical","Industrial","Sports & Entertainment"]
DELIVERY_METHODS = ["CM at Risk","Design-Build","Design-Bid-Build","IPD","CM Agency"]
REGIONS = ["Kansas City","Denver","Dallas","Houston","Atlanta","Phoenix",
    "Portland","Nashville","Minneapolis","Omaha","Austin","Charlotte"]
CSI_DIVISIONS = {"01":"General Requirements","02":"Existing Conditions","03":"Concrete",
    "04":"Masonry","05":"Metals","06":"Wood, Plastics & Composites","07":"Thermal & Moisture Protection",
    "08":"Openings","09":"Finishes","21":"Fire Suppression","22":"Plumbing","23":"HVAC",
    "26":"Electrical","27":"Communications","31":"Earthwork","32":"Exterior Improvements","33":"Utilities"}
TRADES = ["Carpenter","Electrician","Plumber","Ironworker","Laborer","Operator",
    "Concrete Finisher","HVAC Tech","Foreman","Superintendent"]
INCIDENT_TYPES = ["Slip/Trip/Fall","Struck-by","Caught-between","Electrical","Fall from Height",
    "Laceration","Strain/Sprain","Near Miss","Equipment Damage","Heat Illness"]
SEVERITY = ["First Aid","Recordable","Lost Time","Near Miss"]

# ID helper: prefix every key with the batch tag so it can't collide with prior batches
def pid(i):  return f"{BATCH_ID}-PRJ-{i:05d}"

def maybe_missing(v, p=0.05): return v if random.random() > p else None
def messy_date(dt, p_format=0.3, p_impossible=0.02):
    if dt is None: return None
    if random.random() < p_impossible:
        return random.choice(["2099-13-01","0001-01-01","13/45/2021",""])
    fmts = ["%Y-%m-%d","%m/%d/%Y","%d-%b-%Y","%m-%d-%y","%Y/%m/%d","%b %d, %Y"]
    weights = [0.5,0.2,0.1,0.1,0.05,0.05] if random.random() < p_format else [1,0,0,0,0,0]
    return dt.strftime(random.choices(fmts, weights=weights)[0])
def dirty_name(name, p=0.25):
    if random.random() > p: return name
    return random.choice([name.upper(),name.lower(),f"  {name} ",name.replace(",",""),
        name.replace("LLC","L.L.C."),name.replace("Inc","Inc."),name+" ",name.replace(" ","  ")])

def gen_projects(n):
    rows = []
    for i in range(1, n+1):
        ptype = random.choice(PROJECT_TYPES)
        base = np.random.lognormal(16.3, 0.9)
        if ptype in ("Healthcare","Data Center","Mission Critical"): base *= random.uniform(1.5,3.0)
        cv = round(base, 2)
        if random.random() < 0.02: cv = random.choice([-cv, 0, cv*50])
        start = fake.date_between(start_date="-5y", end_date="-6M")
        pdur = random.randint(180,1100); pend = start + timedelta(days=pdur)
        drift = int(np.random.normal(pdur*0.08, pdur*0.12))
        aend = pend + timedelta(days=max(drift,-30))
        status = random.choices(["Active","Complete","On Hold","Closed"], weights=[0.35,0.5,0.05,0.1])[0]
        rows.append({"project_id":pid(i),
            "project_name":maybe_missing(f"{fake.company()} {ptype} Facility",0.02),
            "project_type":maybe_missing(ptype,0.03),"region":random.choice(REGIONS),
            "delivery_method":random.choice(DELIVERY_METHODS),"contract_value":cv,
            "start_date":messy_date(start),"planned_end_date":messy_date(pend),
            "actual_end_date":messy_date(aend) if status in ("Complete","Closed") else None,
            "square_footage":maybe_missing(random.randint(15000,900000),0.04),
            "status":status,"project_manager":maybe_missing(fake.name(),0.03)})
    return pd.DataFrame(rows)   # note: no duplicate injection in incremental batches

def gen_costs(projects):
    rows, lid = [], 1
    for _, p in projects.iterrows():
        cv = p["contract_value"]
        if not isinstance(cv,(int,float)) or cv <= 0: cv = random.uniform(5e6,3e7)
        divs = random.sample(list(CSI_DIVISIONS), k=random.randint(6,len(CSI_DIVISIONS)))
        w = np.random.dirichlet(np.ones(len(divs)))
        for code, frac in zip(divs, w):
            budget = round(cv*frac,2); actual = round(budget*random.uniform(0.85,1.4),2)
            co = round(budget*random.uniform(0,0.15),2) if random.random()<0.4 else 0.0
            if random.random()<0.03: actual = random.choice([None,-actual,0])
            rows.append({"line_item_id":f"{BATCH_ID}-CLI-{lid:07d}","project_id":p["project_id"],
                "csi_division":code,"division_name":CSI_DIVISIONS[code],
                "budget_amount":maybe_missing(budget,0.02),"actual_amount":actual,
                "change_order_amount":co,"cost_code_notes":maybe_missing(fake.sentence(nb_words=4),0.7)})
            lid += 1
    return pd.DataFrame(rows)

def gen_schedule(projects):
    rows, tid = [], 1
    tnames = ["Mobilization","Site Prep","Foundations","Structural Steel","Concrete Pour","Roofing",
        "MEP Rough-in","Drywall","Finishes","Commissioning","Punch List","Substantial Completion"]
    for _, p in projects.iterrows():
        start = pd.to_datetime(p["start_date"], errors="coerce")
        if pd.isna(start): start = datetime.now() - timedelta(days=random.randint(200,800))
        cursor, prev = start, None
        for tn in tnames:
            pdur = random.randint(10,90); adur = max(1,int(np.random.normal(pdur*1.05,pdur*0.2)))
            rows.append({"task_id":f"{BATCH_ID}-TSK-{tid:07d}","project_id":p["project_id"],
                "task_name":tn,"predecessor_task":prev,"planned_start":messy_date(cursor,p_impossible=0.0),
                "planned_duration_days":pdur,"actual_duration_days":maybe_missing(adur,0.05),
                "percent_complete":maybe_missing(random.choice([0,25,50,75,100,random.randint(0,100)]),0.03)})
            prev, cursor = tn, cursor+timedelta(days=adur); tid += 1
    return pd.DataFrame(rows)

def gen_labor(projects):
    rows, wid = [], 1
    for _, p in projects.iterrows():
        for _ in range(random.randint(15,40)):
            rows.append({"timesheet_id":f"{BATCH_ID}-TS-{wid:08d}","project_id":p["project_id"],
                "worker_name":maybe_missing(fake.name(),0.04),"trade":random.choice(TRADES),
                "work_date":messy_date(fake.date_between(start_date="-3y",end_date="today")),
                "regular_hours":random.choice([8,8,8,10,4,12]),
                "overtime_hours":random.choice([0,0,0,2,4,random.randint(0,6)]),
                "hourly_rate":maybe_missing(round(random.uniform(28,78),2),0.03)}); wid += 1
    return pd.DataFrame(rows)

def gen_subs(n=30):
    rows = []
    for i in range(1, n+1):
        raw = f"{fake.company()} {random.choice(['LLC','Inc','Contractors','Construction','Mechanical','Electric Co'])}"
        rows.append({"sub_id":f"{BATCH_ID}-SUB-{i:04d}","sub_name":dirty_name(raw),
            "trade_focus":random.choice(list(CSI_DIVISIONS.values())),"region":random.choice(REGIONS),
            "performance_rating":maybe_missing(round(random.uniform(2.0,5.0),1),0.08),
            "prequalified":random.choice(["Y","N","Yes","No","TRUE","FALSE",None])})
    return pd.DataFrame(rows)

def gen_bids(projects, subs):
    rows, bid = [], 1; sids = subs["sub_id"].tolist()
    for _, p in projects.iterrows():
        for code in random.sample(list(CSI_DIVISIONS), k=random.randint(3,8)):
            for _ in range(random.randint(1,5)):
                rows.append({"bid_id":f"{BATCH_ID}-BID-{bid:07d}","project_id":p["project_id"],
                    "sub_id":random.choice(sids),"csi_division":code,
                    "bid_amount":round(np.random.lognormal(14.5,0.6),2),
                    "awarded":random.choice(["Y","N","N","N"]),
                    "bid_date":messy_date(fake.date_between(start_date="-4y",end_date="today"))}); bid += 1
    return pd.DataFrame(rows)

def gen_safety(projects):
    rows, iid = [], 1
    for _, p in projects.iterrows():
        for _ in range(np.random.poisson(1.5)):
            sev = random.choices(SEVERITY, weights=[0.5,0.25,0.1,0.15])[0]
            rows.append({"incident_id":f"{BATCH_ID}-INC-{iid:06d}","project_id":p["project_id"],
                "incident_date":messy_date(fake.date_between(start_date="-3y",end_date="today")),
                "incident_type":random.choice(INCIDENT_TYPES),"severity":sev,
                "trade_involved":random.choice(TRADES),
                "lost_days":maybe_missing(random.randint(0,30) if sev=="Lost Time" else 0,0.05),
                "description":maybe_missing(fake.sentence(nb_words=8),0.3),
                "root_cause":maybe_missing(random.choice(["Housekeeping","PPE","Training","Equipment","Weather","Unknown"]),0.2)}); iid += 1
    return pd.DataFrame(rows)

StatementMeta(, 6118298a-33bb-43a3-9cda-5976b95a5e7a, 29, Finished, Available, Finished, False)

#### Cell 4

In [10]:
# Generate the batch and land raw CSVs in the dated bronze folder
projects = gen_projects(N_PROJECTS)
subs     = gen_subs()
tables_raw = {
    "projects": projects, "subcontractors": subs, "cost_line_items": gen_costs(projects),
    "schedule_tasks": gen_schedule(projects), "labor_timesheets": gen_labor(projects),
    "sub_bids": gen_bids(projects, subs), "safety_incidents": gen_safety(projects),
}
for name, df in tables_raw.items():
    df.to_csv(f"{BRONZE_BATCH_LOCAL}/{name}.csv", index=False)
    print(f"{name:20s} {len(df):>6,} rows")
print(f"\nBatch {BATCH_ID} landed in {BRONZE_BATCH}")

StatementMeta(, 6118298a-33bb-43a3-9cda-5976b95a5e7a, 30, Finished, Available, Finished, False)

projects                 40 rows
subcontractors           30 rows
cost_line_items         475 rows
schedule_tasks          480 rows
labor_timesheets      1,106 rows
sub_bids                664 rows
safety_incidents         59 rows

Batch B2 landed in Files/bronze/construction_raw/batch_B2


#### Cell 5

In [11]:
# Generate the batch and land raw CSVs in the dated bronze folder
projects = gen_projects(N_PROJECTS)
subs     = gen_subs()
tables_raw = {
    "projects": projects, "subcontractors": subs, "cost_line_items": gen_costs(projects),
    "schedule_tasks": gen_schedule(projects), "labor_timesheets": gen_labor(projects),
    "sub_bids": gen_bids(projects, subs), "safety_incidents": gen_safety(projects),
}
for name, df in tables_raw.items():
    df.to_csv(f"{BRONZE_BATCH_LOCAL}/{name}.csv", index=False)
    print(f"{name:20s} {len(df):>6,} rows")
print(f"\nBatch {BATCH_ID} landed in {BRONZE_BATCH}")

StatementMeta(, 6118298a-33bb-43a3-9cda-5976b95a5e7a, 31, Finished, Available, Finished, False)

projects                 40 rows
subcontractors           30 rows
cost_line_items         450 rows
schedule_tasks          480 rows
labor_timesheets      1,000 rows
sub_bids                624 rows
safety_incidents         54 rows

Batch B2 landed in Files/bronze/construction_raw/batch_B2


#### Cell 5

In [12]:
# Silver cleaning helpers (identical to the silver notebook)
def parse_messy_date(colname):
    c = F.col(colname)
    formats = ["yyyy-MM-dd","MM/dd/yyyy","dd-MMM-yyyy","MM-dd-yy","yyyy/MM/dd","MMM dd, yyyy"]
    parsed = F.coalesce(*[F.try_to_timestamp(c, F.lit(fmt)) for fmt in formats]).cast(T.DateType())
    return F.when((parsed >= F.lit("1990-01-01").cast(T.DateType())) &
                  (parsed <= F.lit("2035-12-31").cast(T.DateType())), parsed).otherwise(None)
def clean_str(colname):
    c = F.trim(F.regexp_replace(F.col(colname), r"\s+", " "))
    return F.when(c=="", None).otherwise(c)
def to_bool(colname):
    c = F.upper(F.trim(F.col(colname).cast("string")))
    return (F.when(c.isin("Y","YES","TRUE","T","1"),F.lit(True))
             .when(c.isin("N","NO","FALSE","F","0"),F.lit(False)).otherwise(None))
def num(colname, cast_to="double"): return F.col(colname).cast(cast_to)

def read_batch(name):
    return spark.read.option("header",True).option("inferSchema",False).csv(f"{BRONZE_BATCH}/{name}.csv")

StatementMeta(, 6118298a-33bb-43a3-9cda-5976b95a5e7a, 32, Finished, Available, Finished, False)

#### Cell 6

In [13]:
# Clean the batch (same transforms as silver)
p = read_batch("projects")
projects_clean = (p
    .withColumn("project_name",clean_str("project_name")).withColumn("project_type",clean_str("project_type"))
    .withColumn("region",clean_str("region")).withColumn("delivery_method",clean_str("delivery_method"))
    .withColumn("project_manager",clean_str("project_manager")).withColumn("contract_value",num("contract_value"))
    .withColumn("square_footage",num("square_footage","int"))
    .withColumn("start_date",parse_messy_date("start_date"))
    .withColumn("planned_end_date",parse_messy_date("planned_end_date"))
    .withColumn("actual_end_date",parse_messy_date("actual_end_date"))
    .withColumn("contract_value_valid",F.when(F.col("contract_value")>0,True).otherwise(False))
    .dropDuplicates(["project_id"]))

s = read_batch("subcontractors")
sn = (s.withColumn("sub_name",clean_str("sub_name")).withColumn("trade_focus",clean_str("trade_focus"))
    .withColumn("region",clean_str("region")).withColumn("performance_rating",num("performance_rating"))
    .withColumn("prequalified",to_bool("prequalified"))
    .withColumn("match_key",F.regexp_replace(F.regexp_replace(F.upper(F.trim(F.col("sub_name"))),r"[.,]",""),r"\s+"," "))
    .withColumn("match_key",F.trim(F.regexp_replace(F.col("match_key"),r"\bL L C\b","LLC"))))
w = Window.partitionBy("match_key").orderBy(F.col("performance_rating").desc_nulls_last(),F.col("sub_id").asc())
subs_clean = sn.withColumn("rn",F.row_number().over(w)).filter(F.col("rn")==1).drop("rn","match_key")

c = read_batch("cost_line_items")
costs_clean = (c.withColumn("csi_division",clean_str("csi_division")).withColumn("division_name",clean_str("division_name"))
    .withColumn("cost_code_notes",clean_str("cost_code_notes")).withColumn("budget_amount",num("budget_amount"))
    .withColumn("actual_amount",num("actual_amount")).withColumn("change_order_amount",num("change_order_amount"))
    .withColumn("actual_valid",F.when(F.col("actual_amount")>0,True).otherwise(False))
    .withColumn("variance",F.when((F.col("budget_amount")>0)&(F.col("actual_amount")>0),F.col("actual_amount")-F.col("budget_amount")))
    .withColumn("overrun_ratio",F.when((F.col("budget_amount")>0)&(F.col("actual_amount")>0),F.round(F.col("actual_amount")/F.col("budget_amount"),4)))
    .dropDuplicates(["line_item_id"]))

sched = (read_batch("schedule_tasks").withColumn("task_name",clean_str("task_name"))
    .withColumn("predecessor_task",clean_str("predecessor_task")).withColumn("planned_start",parse_messy_date("planned_start"))
    .withColumn("planned_duration_days",num("planned_duration_days","int")).withColumn("actual_duration_days",num("actual_duration_days","int"))
    .withColumn("percent_complete",num("percent_complete","int"))
    .withColumn("duration_slip_days",F.col("actual_duration_days")-F.col("planned_duration_days")).dropDuplicates(["task_id"]))

labor = (read_batch("labor_timesheets").withColumn("worker_name",clean_str("worker_name")).withColumn("trade",clean_str("trade"))
    .withColumn("work_date",parse_messy_date("work_date")).withColumn("regular_hours",num("regular_hours"))
    .withColumn("overtime_hours",num("overtime_hours")).withColumn("hourly_rate",num("hourly_rate"))
    .withColumn("total_cost",F.round((F.coalesce(F.col("regular_hours"),F.lit(0))+F.coalesce(F.col("overtime_hours"),F.lit(0))*F.lit(1.5))*F.col("hourly_rate"),2))
    .dropDuplicates(["timesheet_id"]))

bids = (read_batch("sub_bids").withColumn("csi_division",clean_str("csi_division")).withColumn("bid_amount",num("bid_amount"))
    .withColumn("awarded",to_bool("awarded")).withColumn("bid_date",parse_messy_date("bid_date")).dropDuplicates(["bid_id"]))

safety = (read_batch("safety_incidents").withColumn("incident_type",clean_str("incident_type")).withColumn("severity",clean_str("severity"))
    .withColumn("trade_involved",clean_str("trade_involved")).withColumn("root_cause",clean_str("root_cause"))
    .withColumn("description",clean_str("description")).withColumn("incident_date",parse_messy_date("incident_date"))
    .withColumn("lost_days",num("lost_days","int")).withColumn("is_lost_time",F.when(F.col("severity")=="Lost Time",True).otherwise(False))
    .dropDuplicates(["incident_id"]))

print(f"Batch {BATCH_ID} cleaned: {projects_clean.count()} projects, {costs_clean.count()} cost lines, {safety.count()} incidents")

StatementMeta(, 6118298a-33bb-43a3-9cda-5976b95a5e7a, 33, Finished, Available, Finished, False)

Batch B2 cleaned: 40 projects, 450 cost lines, 54 incidents


#### Cell 7

In [14]:
# Stamp lineage, load into silver tables, then dedup safety net
# Incremental batches are stamped as TEST data (data_split="test") so the ML
# notebook can hold them out: train on the original B1 load, evaluate on later
# arrivals -- a provenance-based split that mimics scoring genuinely unseen data.
def add_lineage(df):
    return (df
        .withColumn("batch_id",    F.lit(BATCH_ID))
        .withColumn("ingested_at", F.current_timestamp())
        .withColumn("data_split",  F.lit("test")))

batch = {
    "silver_projects":        (add_lineage(projects_clean), "project_id"),
    "silver_subcontractors":  (add_lineage(subs_clean), "sub_id"),
    "silver_cost_line_items": (add_lineage(costs_clean), "line_item_id"),
    "silver_schedule_tasks":  (add_lineage(sched), "task_id"),
    "silver_labor_timesheets": (add_lineage(labor), "timesheet_id"),
    "silver_sub_bids":        (add_lineage(bids), "bid_id"),
    "silver_safety_incidents": (add_lineage(safety), "incident_id"),
}

for tbl, (df, key) in batch.items():
    # create-or-append: if the silver table doesn't exist yet (e.g. silver
    # notebook hasn't been run), create it from this batch; otherwise append.
    # spark.catalog.tableExists guards against the append-on-missing-table error.
    if spark.catalog.tableExists(tbl):
        mode = "append"
    else:
        mode = "overwrite"   # first write creates the table
        print(f"  (note: {tbl} did not exist -- creating it from this batch)")
    df.write.format("delta").mode(mode).option("mergeSchema","true").saveAsTable(tbl)

    # dedup safety net: if this batch was ever run before, collapse on the key
    # so re-running the same batch can't create duplicates (idempotent).
    deduped = spark.table(tbl).dropDuplicates([key])
    deduped.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(tbl)
    print(f"{tbl:26s} -> now {spark.table(tbl).count():,} total rows")

print(f"\nBatch {BATCH_ID} loaded into silver. Re-running this batch is safe (dedup on key).")

StatementMeta(, 6118298a-33bb-43a3-9cda-5976b95a5e7a, 34, Finished, Available, Finished, False)

silver_projects            -> now 160 total rows
silver_subcontractors      -> now 110 total rows
silver_cost_line_items     -> now 1,829 total rows
silver_schedule_tasks      -> now 1,920 total rows
silver_labor_timesheets    -> now 4,411 total rows
silver_sub_bids            -> now 2,541 total rows
silver_safety_incidents    -> now 243 total rows

Batch B2 loaded into silver. Re-running this batch is safe (dedup on key).


#### Cell 8

In [15]:
# Verify the new batch is queryable alongside the originals
print(f"=== Rows contributed by batch {BATCH_ID} ===\n")
n_new = spark.table("silver_projects").filter(F.col("project_id").startswith(f"{BATCH_ID}-")).count()
n_tot = spark.table("silver_projects").count()
print(f"silver_projects: {n_new} from batch {BATCH_ID}, {n_tot} total")

# batch data has same shape -> overrun stats should sit in a similar range to the originals
print(f"\nOverrun ratio: original vs batch {BATCH_ID}")
(spark.table("silver_cost_line_items")
    .filter(F.col("actual_valid"))
    .withColumn("source", F.when(F.col("line_item_id").startswith(f"{BATCH_ID}-"), f"batch_{BATCH_ID}").otherwise("original"))
    .groupBy("source")
    .agg(F.round(F.mean("overrun_ratio"),3).alias("avg_overrun"), F.count("*").alias("n"))
    .show(truncate=False))


StatementMeta(, 6118298a-33bb-43a3-9cda-5976b95a5e7a, 35, Finished, Available, Finished, False)

=== Rows contributed by batch B2 ===

silver_projects: 40 from batch B2, 160 total

Overrun ratio: original vs batch B2
+--------+-----------+----+
|source  |avg_overrun|n   |
+--------+-----------+----+
|batch_B2|1.131      |436 |
|original|1.186      |1329|
+--------+-----------+----+

